## pathway analysis (ORA analysis)
## TF enrichment, transcriptome sample enrichment

In [ ]:
library(dplyr)
library(tidyverse)
library(data.table)
library(ggplot2)
library(RColorBrewer)
library(genefilter)
library(sva)
library("edgeR")
options(stringsAsFactors=F)
options(scipen=100)

In [ ]:
# ---- Set the following paths to match your environment ----
data_path  = "/path/to/data"            # directory for data
meta_file  = "/path/to/metadata.txt"    # sample metadata, including columns of "id" for sample id and "cluster" for cluster id
# -----------------------------------------------------------

#CV_total_number=10

i=10000

    
out_f=paste0(data_path,"/result/gene_level_projection")
dir.create(out_f, recursive = T)
TF_df=read.table(paste0(data_path, "/intersect_gene_TF.txt"), header=1,row.names=1)
transcriptome_df=read.table(paste0(data_path, "/intersect_gene_TMM_logCPM.txt"), header=1, row.names=1)


transcriptome_meta = read.table(meta_file, header=1)
id_df=data.frame(id=colnames(transcriptome_df))
anno_df=left_join(id_df, transcriptome_meta, by="id")


transcriptome_df_2   = transcriptome_df %>% mutate(var=apply(.,1,var))
transcriptome_df_3   = transcriptome_df_2[order(transcriptome_df_2$var,decreasing=T),] %>% .[1:10000,] %>% dplyr::select(-var)

TF_df_2   = TF_df %>% mutate(var=apply(.,1,var))
TF_df_3   = TF_df_2[order(TF_df_2$var,decreasing=T),] %>% .[1:10000,] %>% dplyr::select(-var)

highly_variable_genes=intersect(rownames(TF_df_3), rownames(transcriptome_df_3))
TF_df_highly=TF_df[highly_variable_genes,]
transcriptome_df_highly=transcriptome_df[highly_variable_genes,]
length(highly_variable_genes)

TF_max=apply(TF_df_highly,2,max)
TF_min=apply(TF_df_highly,2,min)

TF_allzero=TF_max[TF_max==0]
TF_allzero_names=names(TF_allzero)

TF_df_highly_nonzero=TF_df_highly %>% select(-TF_allzero_names)


X <- as.matrix(TF_df_highly_nonzero)
Y <- as.matrix(transcriptome_df_highly)


scaled_X= apply(X,1,function(x){ (x - mean(x))/sd(x) }) 
scaled_Y= apply(Y,1,function(x){ (x - mean(x))/sd(x) })
scaled_X2= apply(scaled_X,1,function(x){ (x - mean(x))/sd(x) }) 
scaled_Y2= apply(scaled_Y,1,function(x){ (x - mean(x))/sd(x) })
scaled_Y2_t=t(scaled_Y2)
scaled_X2_t=t(scaled_X2)


NA_colnames <-as.data.frame(scaled_X2_t) %>%
  select_if(function(x) any(is.na(x))) %>%
  colnames()
length(NA_colnames)

scaled_X3_t=as.data.frame(scaled_X2_t) %>% dplyr::select(-all_of(NA_colnames)) %>% as.matrix()
scaled_Y3=scaled_Y2[colnames(scaled_X3_t),]
scaled_Y3_t=t(scaled_Y3)

XtY=scaled_X3_t %*% scaled_Y3

# SVD
svd_res=svd(XtY)





u=svd_res$u[,1:10]
X_u_df=as.data.frame(u)

colnames(X_u_df)=c("CC1_rotation", "CC2_rotation", "CC3_rotation", "CC4_rotation", "CC5_rotation", "CC6_rotation", "CC7_rotation", "CC8_rotation", "CC9_rotation", "CC10_rotation")
X_u_df$id=colnames(TF_df_highly_nonzero)

X_u_df2=X_u_df
v=svd_res$v[,1:10]
Y_v_df=as.data.frame(v)

colnames(Y_v_df)=c("CC1_rotation", "CC2_rotation", "CC3_rotation", "CC4_rotation", "CC5_rotation", "CC6_rotation", "CC7_rotation", "CC8_rotation", "CC9_rotation", "CC10_rotation")
Y_v_df$id=colnames(transcriptome_df_highly)

all_svd_vec=svd_res$d
rownames(Y_v_df)=Y_v_df$id

Y_v_df2=Y_v_df %>% dplyr::select(-id)
mal_mat=scaled_Y3 %*% as.matrix(Y_v_df2)


mal_df=as.data.frame(mal_mat)
write.table(data.frame(gene=rownames(mal_df),mal_df),paste0(out_f, "/transcriptome_CCA_bias_vector.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)

mal_df=read.table(paste0(out_f, "/transcriptome_CCA_bias_vector.txt"),header=1, row.names=1)

In [ ]:

##ORA
library(clusterProfiler)
library(DOSE)
library(org.Hs.eg.db)

res=mal_df
out_f2=paste0(out_f, "/ORA")
dir.create(out_f2)


for (i in seq(10)){
    CC_df=data.frame(genes=rownames(res),CC=res[,i])
    annotation_df <- select(org.Hs.eg.db,
              keys = CC_df$genes,
              keytype = "ENSEMBL",
              columns = c("ENSEMBL", "ENTREZID"))
    colnames(annotation_df)=c("genes", "geneid")
    CC_df2=left_join(CC_df, annotation_df, by="genes") 
    CC_df3=CC_df2 %>% na.omit()
    CC_df4=CC_df3[!(duplicated(CC_df3$geneid) | duplicated(CC_df3$geneid, fromLast = TRUE)),]

    gene_num=nrow(CC_df4)
    tenpercent_gene_num=gene_num%/%10
    CC_df_order=CC_df4[order(CC_df4$CC,decreasing=TRUE),]
    gene_list=CC_df_order[1:tenpercent_gene_num,"geneid"]
    background_gene_list=CC_df4$geneid
    
    kk2 <- enrichKEGG(gene     = gene_list, 
                         universe = background_gene_list, 
                         organism     = 'hsa',
                         maxGSSize    = 2000, 
                         pvalueCutoff = 1,
                         qvalueCutoff = 1)
    res_df=as.data.frame(kk2)
    write.table(data.frame(pathway=rownames(res_df),res_df),paste0(out_f2, "/CC", i, "_top_KEGG_ORA_ver3.txt"),row.names=FALSE,col.names=TRUE, sep="!", append=F, quote=F)
    res_df$pvalue=as.numeric(res_df$pvalue)
    res_df$minuslog10pvalue=-log10(res_df$pvalue)  
    res_df_top_order=res_df[order(res_df$minuslog10pvalue,decreasing=TRUE), ]
    res_df_top_order_top20=res_df_top_order[1:20,]
    res_df_top_order_top20_reverse=res_df_top_order_top20[order(res_df_top_order_top20$minuslog10pvalue, decreasing=FALSE),]
    res_df_top_order_top20$Description=factor(res_df_top_order_top20$Description, levels=res_df_top_order_top20_reverse$Description)
    kegg_data <- download_KEGG(species = "hsa", keyType = "kegg")
    pathway_gene_n=kegg_data[[1]] %>% group_by(from) %>% summarise(count = n())
    maxGSSize_fil_df=pathway_gene_n %>% filter(count<=2000)
    print(paste("Number of KEGG pathways with maxGSSize <= 2000:", nrow(maxGSSize_fil_df)))
    pathway_num=nrow(maxGSSize_fil_df)
    g <- ggplot(res_df_top_order_top20, aes(Description, minuslog10pvalue))
    g + geom_bar(stat = "identity") + theme(axis.text.x  = element_text(angle = 90, size = 12), text = element_text(size = 24))+
    theme_bw()+ coord_flip()+ geom_hline(yintercept=-log10(0.05/pathway_num),size=0.5,color='red')
    ggsave(paste0(out_f2, "/CC", i, "_top_KEGG_ORA.pdf"), width=10, height=10)

    CC_df_order=CC_df4[order(CC_df4$CC, decreasing=FALSE),]
    gene_list=CC_df_order[1:tenpercent_gene_num,"geneid"]
    
    
    kk2 <- enrichKEGG(gene     = gene_list, 
                         universe = background_gene_list, 
                         organism     = 'hsa',
                         maxGSSize    = 2000, 
                         pvalueCutoff = 1,
                         qvalueCutoff = 1)
    res_df=as.data.frame(kk2)
    write.table(data.frame(pathway=rownames(res_df),res_df),paste0(out_f2, "/CC", i, "_bottom_KEGG_ORA_ver3.txt"),row.names=FALSE,col.names=TRUE, sep="!", append=F, quote=F)
    res_df$pvalue=as.numeric(res_df$pvalue)
    res_df$minuslog10pvalue=-log10(res_df$pvalue)  
    res_df_bottom_order=res_df[order(res_df$minuslog10pvalue,decreasing=TRUE), ]
    res_df_bottom_order_bottom20=res_df_bottom_order[1:20,]
    res_df_bottom_order_bottom20_reverse=res_df_bottom_order_bottom20[order(res_df_bottom_order_bottom20$minuslog10pvalue, decreasing=FALSE),]
    res_df_bottom_order_bottom20$Description=factor(res_df_bottom_order_bottom20$Description, levels=res_df_bottom_order_bottom20_reverse$Description)
    g <- ggplot(res_df_bottom_order_bottom20, aes(Description, minuslog10pvalue))
    g + geom_bar(stat = "identity") + theme(axis.text.x  = element_text(angle = 90, size = 12), text = element_text(size = 24))+
    theme_bw()+ coord_flip()+ geom_hline(yintercept=-log10(0.05/pathway_num),size=0.5,color='red')
    ggsave(paste0(out_f2, "/CC", i, "bottom_KEGG_ORA.pdf"), width=10, height=10)
}
## MsigDB
library(msigdbr)

msig_ORA_fun=function(i, top_or_bottom, gene_list, background_gene_list, category_n){
    out_f3=paste0(out_f2, "/",category_n)
    dir.create(out_f3)
    pathway_df = msigdbr(species="Homo sapiens",category=category_n) %>% 
           dplyr::select(gs_name,entrez_gene)
    kk2 <- enricher(gene=gene_list,universe=background_gene_list,TERM2GENE=pathway_df,maxGSSize=1000,pvalueCutoff=1,qvalueCutoff=1)
    res_df=as.data.frame(kk2)
    write.table(data.frame(pathway=rownames(res_df),res_df),paste0(out_f3, "/CC", i, "_",top_or_bottom, "_", category_n, "_ORA_logp.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)
    res_df$pvalue=as.numeric(res_df$pvalue)
    res_df$minuslog10pvalue=-log10(res_df$pvalue)  
    res_df_top_order=res_df[order(res_df$minuslog10pvalue,decreasing=TRUE), ]
    res_df_top_order_top20=res_df_top_order[1:20,]
    res_df_top_order_top20_reverse=res_df_top_order_top20[order(res_df_top_order_top20$minuslog10pvalue, decreasing=FALSE),]
    res_df_top_order_top20$Description=factor(res_df_top_order_top20$Description, levels=res_df_top_order_top20_reverse$Description)
    pathway_gene_n=as.data.frame(pathway_df) %>% group_by(gs_name) %>% summarise(count = n()) 
    maxGSSize_fil_df=pathway_gene_n %>% filter(count<=1000)
    print(paste("Number of", category_n, " pathways with maxGSSize <= 1000:", nrow(maxGSSize_fil_df)))
    pathway_num=nrow(maxGSSize_fil_df)
    g <- ggplot(res_df_top_order_top20, aes(Description, minuslog10pvalue))
    
    g + geom_bar(stat = "identity") + theme(axis.text.x  = element_text(angle = 90, size = 12), text = element_text(size = 24))+
    theme_bw()+ coord_flip()+ geom_hline(yintercept=-log10(0.05/pathway_num),size=0.5,color='red')

    ggsave(paste0(out_f3, "/CC", i, "_", top_or_bottom, "_", category_n, "_ORA_logp.pdf"), width=10, height=10)
}

msig_ORA_C3_fun=function(i, top_or_bottom, gene_list, background_gene_list, subcategory_n){
    out_f3=paste0(out_f2, "/C3_",subcategory_n)
    dir.create(out_f3)
    pathway_df = msigdbr(species="Homo sapiens",category="C3",subcategory = subcategory_n) %>% 
           dplyr::select(gs_name,entrez_gene)
    kk2 <- enricher(gene=gene_list,universe=background_gene_list,TERM2GENE=pathway_df,maxGSSize=1000,pvalueCutoff=1,qvalueCutoff=1)
    res_df=as.data.frame(kk2)
    write.table(data.frame(pathway=rownames(res_df),res_df),paste0(out_f3, "/CC", i, "_", top_or_bottom, "_C3_", subcategory_n, "_ORA_logp.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)
    res_df$pvalue=as.numeric(res_df$pvalue)
    res_df$minuslog10pvalue=-log10(res_df$pvalue)  
    res_df_top_order=res_df[order(res_df$minuslog10pvalue,decreasing=TRUE), ]
    res_df_top_order_top20=res_df_top_order[1:20,]
    res_df_top_order_top20_reverse=res_df_top_order_top20[order(res_df_top_order_top20$minuslog10pvalue, decreasing=FALSE),]
    res_df_top_order_top20$Description=factor(res_df_top_order_top20$Description, levels=res_df_top_order_top20_reverse$Description)
    pathway_gene_n=as.data.frame(pathway_df) %>% group_by(gs_name) %>% summarise(count = n()) 
    maxGSSize_fil_df=pathway_gene_n %>% filter(count<=1000)
    print(paste("Number of", subcategory_n, " pathways with maxGSSize <= 1000:", nrow(maxGSSize_fil_df)))
    pathway_num=nrow(maxGSSize_fil_df)
    g <- ggplot(res_df_top_order_top20, aes(Description, minuslog10pvalue))
    
    g + geom_bar(stat = "identity") + theme(axis.text.x  = element_text(angle = 90, size = 12), text = element_text(size = 24))+
    theme_bw()+ coord_flip()+ geom_hline(yintercept=-log10(0.05/pathway_num),size=0.5,color='red')

    ggsave(paste0(out_f3, "/CC", i, "_", top_or_bottom, "_C3_", subcategory_n, "_ORA_logp.pdf"), width=10, height=10)
}


for (i in seq(10)){
    CC_df=data.frame(genes=rownames(res),CC=res[,i])
    annotation_df <- select(org.Hs.eg.db,
              keys = CC_df$gene,
              keytype = "ENSEMBL",
              columns = c("ENSEMBL", "ENTREZID"))
    colnames(annotation_df)=c("genes", "geneid")
    CC_df2=left_join(CC_df, annotation_df, by="genes") 
    CC_df3=CC_df2 %>% na.omit()
    CC_df4=CC_df3[!(duplicated(CC_df3$geneid) | duplicated(CC_df3$geneid, fromLast = TRUE)),]

    gene_num=nrow(CC_df4)
    tenpercent_gene_num=gene_num%/%10
    CC_df_order=CC_df4[order(CC_df4$CC,decreasing=TRUE),]
    gene_list=CC_df_order[1:tenpercent_gene_num,"geneid"]
    background_gene_list=CC_df4$geneid
    top_or_bottom="top"
    msig_ORA_fun(i, top_or_bottom, gene_list, background_gene_list, "H")
    msig_ORA_fun(i, top_or_bottom, gene_list, background_gene_list, "C2")
    msig_ORA_fun(i, top_or_bottom, gene_list, background_gene_list,"C4")
    msig_ORA_fun(i, top_or_bottom, gene_list, background_gene_list,"C5")
    msig_ORA_fun(i, top_or_bottom, gene_list, background_gene_list,"C6")
    msig_ORA_fun(i, top_or_bottom, gene_list, background_gene_list,"C7")
    msig_ORA_fun(i, top_or_bottom, gene_list, background_gene_list,"C8")
    msig_ORA_C3_fun(i, top_or_bottom, gene_list, background_gene_list,"TFT:GTRD")
    msig_ORA_C3_fun(i, top_or_bottom, gene_list, background_gene_list,"TFT:TFT_Legacy")
        

    CC_df_order=CC_df4[order(CC_df4$CC, decreasing=FALSE),]
    gene_list=CC_df_order[1:tenpercent_gene_num,"geneid"]
    top_or_bottom="bottom"
    msig_ORA_fun(i, top_or_bottom, gene_list, background_gene_list,"H")
    msig_ORA_fun(i, top_or_bottom, gene_list, background_gene_list,"C2")
    msig_ORA_fun(i, top_or_bottom, gene_list, background_gene_list,"C4")
    msig_ORA_fun(i, top_or_bottom, gene_list, background_gene_list,"C5")
    msig_ORA_fun(i, top_or_bottom, gene_list, background_gene_list,"C6")
    msig_ORA_fun(i, top_or_bottom, gene_list, background_gene_list,"C7")
    msig_ORA_fun(i, top_or_bottom, gene_list, background_gene_list,"C8")
    msig_ORA_C3_fun(i, top_or_bottom, gene_list, background_gene_list,"TFT:GTRD")
    msig_ORA_C3_fun(i, top_or_bottom, gene_list, background_gene_list,"TFT:TFT_Legacy")
    
}

##GSEA
out_f4=paste0(out_f, "/GSEA")
dir.create(out_f4)

for (i in seq(10)){
    CC_df=data.frame(genes=rownames(res),CC=res[,i])
    annotation_df <- select(org.Hs.eg.db,
              keys = CC_df$genes,
              keytype = "ENSEMBL",
              columns = c("ENSEMBL", "ENTREZID"))
    colnames(annotation_df)=c("genes", "geneid")
    CC_df2=left_join(CC_df, annotation_df, by="genes") 
    CC_df3=CC_df2 %>% na.omit()
    CC_df4=CC_df3[!(duplicated(CC_df3$geneid) | duplicated(CC_df3$geneid, fromLast = TRUE)),]
    geneList=CC_df4$CC
    names(geneList)=as.character(CC_df4$geneid)
    geneList = sort(geneList, decreasing = TRUE)
    kk2 <- gseKEGG(geneList = geneList, eps=0, organism = 'hsa', minGSSize = 10, verbose = FALSE)
    res_df=as.data.frame(kk2)
    write.table(data.frame(pathway=rownames(res_df),res_df),paste0(out_f4, "/CC", i, "_GSEA_KEGG_ver3.txt"),row.names=FALSE,col.names=TRUE, sep="!", append=F, quote=F)
    res_df$pvalue=as.numeric(res_df$pvalue)
    res_df$minuslog10pvalue=-log10(res_df$pvalue)  
    res_df_top=res_df %>% filter(enrichmentScore>0)
    res_df_top_order=res_df_top[order(res_df_top$minuslog10pvalue,decreasing=TRUE), ]
    res_df_top_order_top20=res_df_top_order[1:20,]
    res_df_top_order_top20_reverse=res_df_top_order_top20[order(res_df_top_order_top20$minuslog10pvalue, decreasing=FALSE),]
    res_df_top_order_top20$Description=factor(res_df_top_order_top20$Description, levels=res_df_top_order_top20_reverse$Description)
    kegg_data <- download_KEGG(species = "hsa", keyType = "kegg")
    pathway_gene_n=kegg_data[[1]] %>% group_by(from) %>% summarise(count = n())
    minGSSize_fil_df=pathway_gene_n %>% filter(count>=10)
    print(paste("Number of KEGG pathways with minGSSize >= 10:", nrow(minGSSize_fil_df)))
    pathway_num=nrow(minGSSize_fil_df)   
    g <- ggplot(res_df_top_order_top20, aes(Description, minuslog10pvalue))
    g + geom_bar(stat = "identity") + theme(axis.text.x  = element_text(angle = 90, size = 12), text = element_text(size = 24))+
    theme_bw()+ coord_flip()+ geom_hline(yintercept=-log10(0.05/pathway_num),size=0.5,color='red')
    ggsave(paste0(out_f4, "/CC", i, "top_GSEA_KEGG.pdf"), width=10, height=10)

    res_df_bottom=res_df %>% filter(enrichmentScore<0)
    res_df_bottom_order=res_df_bottom[order(res_df_bottom$minuslog10pvalue,decreasing=TRUE), ]
    res_df_bottom_order_bottom20=res_df_bottom_order[1:20,]
    res_df_bottom_order_bottom20_reverse=res_df_bottom_order_bottom20[order(res_df_bottom_order_bottom20$minuslog10pvalue, decreasing=FALSE),]
    res_df_bottom_order_bottom20$Description=factor(res_df_bottom_order_bottom20$Description, levels=res_df_bottom_order_bottom20_reverse$Description)
    g <- ggplot(res_df_bottom_order_bottom20, aes(Description, minuslog10pvalue))
    g + geom_bar(stat = "identity") + theme(axis.text.x  = element_text(angle = 90, size = 12), text = element_text(size = 24))+
    theme_bw()+ coord_flip()+ geom_hline(yintercept=-log10(0.05/pathway_num),size=0.5,color='red')
    ggsave(paste0(out_f4, "/CC", i, "bottom_GSEA_KEGG.pdf"), width=10, height=10)
}

## MsigDB
library(msigdbr)



GSEA_fun=function(i, geneList, category_n){
    pathway_df = msigdbr(species="Homo sapiens",category=category_n) %>% 
           dplyr::select(gs_name,entrez_gene)
    kk2 <- GSEA(geneList, TERM2GENE = pathway_df, eps=0, minGSSize = 10, verbose = FALSE)
    res_df=as.data.frame(kk2)
    out_f5=paste0(out_f4, "/", category_n)
    dir.create(out_f5)
    write.table(data.frame(pathway=rownames(res_df),res_df),paste0(out_f5, "/CC", i, "_GSEA_", category_n, ".txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)
    res_df$pvalue=as.numeric(res_df$pvalue)
    res_df$minuslog10pvalue=-log10(res_df$pvalue)  
    res_df_top=res_df %>% filter(enrichmentScore>0)
    res_df_top_order=res_df_top[order(res_df_top$minuslog10pvalue,decreasing=TRUE), ]
    res_df_top_order_top20=res_df_top_order[1:20,]
    res_df_top_order_top20_reverse=res_df_top_order_top20[order(res_df_top_order_top20$minuslog10pvalue, decreasing=FALSE),]
    res_df_top_order_top20$Description=factor(res_df_top_order_top20$Description, levels=res_df_top_order_top20_reverse$Description)
    pathway_gene_n=as.data.frame(pathway_df) %>% group_by(gs_name) %>% summarise(count = n()) 
minGSSize_fil_df=pathway_gene_n %>% filter(count>=10)
    print(paste("Number of ", category_n, " pathways with minGSSize >= 10:", nrow(minGSSize_fil_df)))
    pathway_num=nrow(minGSSize_fil_df)   
    g <- ggplot(res_df_top_order_top20, aes(Description, minuslog10pvalue))
    g + geom_bar(stat = "identity") + theme(axis.text.x  = element_text(angle = 90, size = 12), text = element_text(size = 24))+
    theme_bw()+ coord_flip()+ geom_hline(yintercept=-log10(0.05/pathway_num),size=0.5,color='red')
    
    ggsave(paste0(out_f5, "/CC", i, "_GSEA_", category_n, "_top_GSEA_logp.pdf"), width=10, height=10)

    res_df_bottom=res_df %>% filter(enrichmentScore<0)
    res_df_bottom_order=res_df_bottom[order(res_df_bottom$minuslog10pvalue,decreasing=TRUE), ]
    res_df_bottom_order_bottom20=res_df_bottom_order[1:20,]
    res_df_bottom_order_bottom20_reverse=res_df_bottom_order_bottom20[order(res_df_bottom_order_bottom20$minuslog10pvalue, decreasing=FALSE),]
    res_df_bottom_order_bottom20$Description=factor(res_df_bottom_order_bottom20$Description, levels=res_df_bottom_order_bottom20_reverse$Description)
    g <- ggplot(res_df_bottom_order_bottom20, aes(Description, minuslog10pvalue))
    g + geom_bar(stat = "identity") + theme(axis.text.x  = element_text(angle = 90, size = 12), text = element_text(size = 24))+
    theme_bw()+ coord_flip()+ geom_hline(yintercept=-log10(0.05/pathway_num),size=0.5,color='red')
    ggsave(paste0(out_f5, "/CC", i, "_GSEA_", category_n,  "_bottom_GSEA_logp.pdf"), width=10, height=10)
}

GSEA_C3_fun=function(i, geneList, subcategory_n){
    pathway_df = msigdbr(species="Homo sapiens",category="C3",subcategory = subcategory_n) %>% 
           dplyr::select(gs_name,entrez_gene)
    kk2 <- GSEA(geneList, TERM2GENE = pathway_df, eps=0, minGSSize = 10, verbose = FALSE)
    res_df=as.data.frame(kk2)
    out_f5=paste0(out_f4, "/C3_", subcategory_n)
    dir.create(out_f5)
    write.table(data.frame(pathway=rownames(res_df),res_df),paste0(out_f5,  "/CC", i, "_", subcategory_n, "_GSEA_logp.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)
    res_df$pvalue=as.numeric(res_df$pvalue)
    res_df$minuslog10pvalue=-log10(res_df$pvalue)  
    res_df_top=res_df %>% filter(enrichmentScore>0)
    res_df_top_order=res_df_top[order(res_df_top$minuslog10pvalue,decreasing=TRUE), ]
    res_df_top_order_top20=res_df_top_order[1:20,]
    res_df_top_order_top20_reverse=res_df_top_order_top20[order(res_df_top_order_top20$minuslog10pvalue, decreasing=FALSE),]
    res_df_top_order_top20$Description=factor(res_df_top_order_top20$Description, levels=res_df_top_order_top20_reverse$Description)
    pathway_gene_n=as.data.frame(pathway_df) %>% group_by(gs_name) %>% summarise(count = n()) 
    minGSSize_fil_df=pathway_gene_n %>% filter(count>=10)
    print(paste("Number of ", subcategory_n, " pathways with minGSSize >= 10:", nrow(minGSSize_fil_df)))
    pathway_num=nrow(minGSSize_fil_df)   
    g <- ggplot(res_df_top_order_top20, aes(Description, minuslog10pvalue))
    g + geom_bar(stat = "identity") + theme(axis.text.x  = element_text(angle = 90, size = 12), text = element_text(size = 24))+
    theme_bw()+ coord_flip()+ geom_hline(yintercept=-log10(0.05/pathway_num),size=0.5,color='red')
    
    ggsave(paste0(out_f5, "/CC", i, "_", subcategory_n, "_top_GSEA_logp.pdf"), width=10, height=10)

    res_df_bottom=res_df %>% filter(enrichmentScore<0)
    res_df_bottom_order=res_df_bottom[order(res_df_bottom$minuslog10pvalue,decreasing=TRUE), ]
    res_df_bottom_order_bottom20=res_df_bottom_order[1:20,]
    res_df_bottom_order_bottom20_reverse=res_df_bottom_order_bottom20[order(res_df_bottom_order_bottom20$minuslog10pvalue, decreasing=FALSE),]
    res_df_bottom_order_bottom20$Description=factor(res_df_bottom_order_bottom20$Description, levels=res_df_bottom_order_bottom20_reverse$Description)
    g <- ggplot(res_df_bottom_order_bottom20, aes(Description, minuslog10pvalue))
    g + geom_bar(stat = "identity") + theme(axis.text.x  = element_text(angle = 90, size = 12), text = element_text(size = 24))+
    theme_bw()+ coord_flip()+ geom_hline(yintercept=-log10(0.05/pathway_num),size=0.5,color='red')
    ggsave(paste0(out_f5, "/CC", i, "_", subcategory_n,  "_bottom_GSEA_logp.pdf"), width=10, height=10)
}


for (i in seq(10)){
    CC_df=data.frame(genes=rownames(res),CC=res[,i])
    annotation_df <- select(org.Hs.eg.db,
              keys = CC_df$genes,
              keytype = "ENSEMBL",
              columns = c("ENSEMBL", "ENTREZID"))
    colnames(annotation_df)=c("genes", "geneid")
    CC_df2=left_join(CC_df, annotation_df, by="genes") 
    CC_df3=CC_df2 %>% na.omit()
    CC_df4=CC_df3[!(duplicated(CC_df3$geneid) | duplicated(CC_df3$geneid, fromLast = TRUE)),]
    geneList=CC_df4$CC
    names(geneList)=as.character(CC_df4$geneid)
    geneList = sort(geneList, decreasing = TRUE)
    
    GSEA_fun(i, geneList, "H")
    GSEA_fun(i, geneList, "C2")
    GSEA_fun(i, geneList, "C4")
    GSEA_fun(i, geneList, "C5")
    GSEA_fun(i, geneList, "C6")
    GSEA_fun(i, geneList, "C7")
    GSEA_fun(i, geneList, "C8")
    GSEA_C3_fun(i, geneList, "TFT:GTRD")
    GSEA_C3_fun(i, geneList, "TFT:TFT_Legacy")
}

##CC Zscore
out_f_Z=paste0(data_path, "/result/CC_Zscored")
dir.create(out_f_Z)

CC_list=c("CC1_rotation", "CC2_rotation", "CC3_rotation", "CC4_rotation", "CC5_rotation", "CC6_rotation", "CC7_rotation", "CC8_rotation", "CC9_rotation", "CC10_rotation")

CC_df=read.table(paste0(data_path, "/result/transcriptome_raw_CC/CC1_rotation_CCscore.txt"))
colnames(CC_df)=c("id", CC_list[1])
for (i in seq(2,10)){
    CC_tmp_df=read.table(paste0(data_path, "/result/transcriptome_raw_CC/", CC_list[i], "_CCscore.txt"))
    colnames(CC_tmp_df)=c("id", CC_list[i])
    CC_df=left_join(CC_df, CC_tmp_df, by="id")
}

rownames(CC_df)=CC_df$id
CC_df2=CC_df %>% dplyr::select(CC_list)    
CC_mat2=as.matrix(CC_df2)

CC_mat3=matrix(as.numeric(CC_mat2), nrow=nrow(CC_mat2))
CC_mat4=matrix(rep(NA, nrow(CC_mat3)*ncol(CC_mat3)), ncol=ncol(CC_mat3))
colnames(CC_mat4)=colnames(CC_mat2)
rownames(CC_mat4)=rownames(CC_mat2)
CC_mean=apply(CC_mat3, 2, mean)
CC_mean=as.numeric(CC_mean)
CC_sd=apply(CC_mat3, 2, sd)
CC_sd=as.numeric(CC_sd)
for (i in seq(10)){
    CC_mat4[, i]=(CC_mat3[,i]-CC_mean[i])/CC_sd[i]
}
CC_res=as.data.frame(CC_mat4) 
CC_res$id=rownames(CC_res)

write.table(CC_res, paste0(out_f_Z, "/CC_Zscored_cluster.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)


#TF
CC_df=read.table(paste0(data_path, "/result/raw_CC/CC1_rotation_CCscore.txt"))
colnames(CC_df)=c("sample_id", CC_list[1])
for (i in seq(2,10)){
    CC_tmp_df=read.table(paste0(data_path, "/result/raw_CC/", CC_list[i], "_CCscore.txt"))
    colnames(CC_tmp_df)=c("sample_id", CC_list[i])
    CC_df=left_join(CC_df, CC_tmp_df, by="sample_id")
}             
rownames(CC_df)=CC_df$sample_id
CC_df2=CC_df %>% dplyr::select(CC_list)          
CC_mat2=as.matrix(CC_df2)

CC_mat3=matrix(as.numeric(CC_mat2), nrow=nrow(CC_mat2))
CC_mat4=matrix(rep(NA, nrow(CC_mat3)*ncol(CC_mat3)), ncol=ncol(CC_mat3))
colnames(CC_mat4)=colnames(CC_mat2)
rownames(CC_mat4)=rownames(CC_mat2)
CC_mean=apply(CC_mat3, 2, mean)
CC_mean=as.numeric(CC_mean)
CC_sd=apply(CC_mat3, 2, sd)
CC_sd=as.numeric(CC_sd)
for (i in seq(10)){
    CC_mat4[, i]=(CC_mat3[,i]-CC_mean[i])/CC_sd[i]
    }
CC_res=as.data.frame(CC_mat4) 
CC_res$id=rownames(CC_res)
write.table(CC_res, paste0(out_f_Z, "/TF_CC_Zscored.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)


TF_df=read.table(paste0(out_f_Z, "/TF_CC_Zscored.txt"), header=1)
transcriptome_df=read.table(paste0(out_f_Z, "/CC_Zscored_cluster.txt"), header=1)

rownames(TF_df)=TF_df$id
TF_df$id2=TF_df$id
TF_df2=TF_df %>% separate(id2, c("GSM", "TF_name"), sep="_")

TF_name_list=unique(TF_df2$TF_name)

threshold_fn=function(i){
    TF_enrichment_df=data.frame(matrix(rep(NA, 8), ncol=8))[0,]
    colnames(TF_enrichment_df)=c("CC_name", "TF","top_TF", "nontop_TF", "top_nonTF", "nontop_nonTF", "odds", "p")
    CC_list=c("CC1_rotation", "CC2_rotation", "CC3_rotation", "CC4_rotation", "CC5_rotation", "CC6_rotation", "CC7_rotation", "CC8_rotation", "CC9_rotation", "CC10_rotation")

    for (CC in CC_list){
    
        TF_df2_order=TF_df2[order(TF_df2[, CC], decreasing=TRUE), ]
        top_list=TF_df2_order[1:i, "TF_name"]
        for (TF_n in TF_name_list){
            num_top=sum(top_list==TF_n)
            all_TF_list=TF_df2$TF_name
            num_all=sum(all_TF_list==TF_n)
            top_TF_n=num_top
            nontop_TF_n=num_all-num_top
            top_nonTF_n=i-num_top
            nontop_nonTF_n=2868-i+num_top-num_all
        
            mat=matrix(c(top_TF_n, top_nonTF_n, nontop_TF_n, nontop_nonTF_n), 2,2)
            testResult <- fisher.test(mat, alternative = "greater")
        
            or <- unname(testResult$estimate) 
            pValue <- unname(testResult$p.value) 
            new_df=data.frame(
                  CC_name=CC,
                  TF=TF_n,
                  top_TF=top_TF_n,
                  nontop_TF=nontop_TF_n,
                  top_nonTF=top_nonTF_n,
                  nontop_nonTF=nontop_nonTF_n,
                  odds=or,
                  p=pValue,
                  stringsAsFactors=FALSE
            )
            TF_enrichment_df=rbind(TF_enrichment_df, new_df)
        }
    }
    out_f_TF=paste0(out_f_Z,"/TF_top",i, "_enrichment_ver2")
    dir.create(out_f_TF, recursive=T)
    write.table(TF_enrichment_df, paste0(out_f_TF, "/TF_enrichment.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F) 
    TF_enrichment_top20=matrix(rep(NA, 20*10), ncol=10)
    colnames(TF_enrichment_top20)=c("CC1_rotation", "CC2_rotation", "CC3_rotation", "CC4_rotation", "CC5_rotation", "CC6_rotation", "CC7_rotation", "CC8_rotation", "CC9_rotation", "CC10_rotation")
    rownames(TF_enrichment_top20)=paste0("top_", seq(20))
    
    for (CC in CC_list){
        TF_enrichment_df_CC=TF_enrichment_df %>% filter(CC_name==CC)
        TF_enrichment_df_CC$minuslog10p=-log10(TF_enrichment_df_CC$p)
        TF_enrichment_CC2=TF_enrichment_df_CC %>% filter(minuslog10p>-log10(0.05/8200))
        TF_enrichment_CC_order=TF_enrichment_CC2[order(TF_enrichment_CC2$odds, decreasing=TRUE), ]
    
        TF_enrichment_top20[,CC]=TF_enrichment_CC_order[1:20, "TF"]
    }
    TF_enrichment_top20_df=as.data.frame(TF_enrichment_top20)
    write.table(data.frame(top=rownames(TF_enrichment_top20_df),TF_enrichment_top20_df), paste0(out_f_TF, "/TF_enrichment_top_odds.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F) 


    TF_enrichment_df_bottom=data.frame(matrix(rep(NA, 8), ncol=8))[0,]
    colnames(TF_enrichment_df_bottom)=c("bottom_TF", "nonbottom_TF", "bottom_nonTF", "nonbottom_nonTF", "odds", "p")
    CC_list=c("CC1_rotation", "CC2_rotation", "CC3_rotation", "CC4_rotation", "CC5_rotation", "CC6_rotation", "CC7_rotation", "CC8_rotation", "CC9_rotation", "CC10_rotation")

    for (CC in CC_list){
    
        TF_df2_order=TF_df2[order(TF_df2[, CC], decreasing=FALSE), ]
        bottom_list=TF_df2_order[1:i, "TF_name"]
        for (TF_n in TF_name_list){
            num_bottom=sum(bottom_list==TF_n)
            all_TF_list=TF_df2$TF_name
            num_all=sum(all_TF_list==TF_n)
            bottom_TF_n=num_bottom
            nonbottom_TF_n=num_all-num_bottom
            bottom_nonTF_n=i-num_bottom
            nonbottom_nonTF_n=2868-i+num_bottom-num_all
        
            mat=matrix(c(bottom_TF_n, bottom_nonTF_n, nonbottom_TF_n, nonbottom_nonTF_n), 2,2)
            testResult <- fisher.test(mat, alternative = "greater")
        
            or <- unname(testResult$estimate) 
            pValue <- unname(testResult$p.value) 
            new_df=data.frame(
              CC_name=CC,
              TF=TF_n,
              bottom_TF=bottom_TF_n,
              nonbottom_TF=nonbottom_TF_n,
              bottom_nonTF=bottom_nonTF_n,
              nonbottom_nonTF=nonbottom_nonTF_n,
              odds=or,
              p=pValue,
              stringsAsFactors=FALSE
            )
            TF_enrichment_df_bottom=rbind(TF_enrichment_df_bottom, new_df)
        }
    }
    out_f_TF_bottom=paste0(out_f_Z,"/TF_bottom",i, "_enrichment_ver2")
    dir.create(out_f_TF_bottom, recursive=T)
    write.table(TF_enrichment_df_bottom, paste0(out_f_TF_bottom, "/TF_enrichment_bottom.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F) 
    TF_enrichment_bottom20=matrix(rep(NA, 20*10), ncol=10)
    colnames(TF_enrichment_bottom20)=c("CC1_rotation", "CC2_rotation", "CC3_rotation", "CC4_rotation", "CC5_rotation", "CC6_rotation", "CC7_rotation", "CC8_rotation", "CC9_rotation", "CC10_rotation")
    rownames(TF_enrichment_bottom20)=paste0("bottom_", seq(20))
    
    for (CC in CC_list){
        TF_enrichment_df_CC=TF_enrichment_df_bottom %>% filter(CC_name==CC)
        TF_enrichment_df_CC$minuslog10p=-log10(TF_enrichment_df_CC$p)
        TF_enrichment_df_CC2=TF_enrichment_df_CC %>% filter(minuslog10p>-log10(0.05/8200))
       
        TF_enrichment_CC_order=TF_enrichment_df_CC2[order(TF_enrichment_df_CC2$odds, decreasing=TRUE), ]
    
        TF_enrichment_bottom20[,CC]=TF_enrichment_CC_order[1:20, "TF"]
    }
    TF_enrichment_bottom20_df=as.data.frame(TF_enrichment_bottom20)
    write.table(data.frame(bottom=rownames(TF_enrichment_bottom20_df),TF_enrichment_bottom20_df), paste0(out_f_TF_bottom, "/TF_enrichment_bottom_odds.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F) 
}

threshold_fn(100)
threshold_fn(200)
threshold_fn(300)
threshold_fn(500)

out_f_Z_transcriptome=paste0(wd, "/result/CC_Zscored_transcriptome")
cluster_list=unique(anno_df$cluster)
transcriptome_df2=left_join(transcriptome_df,anno_df, by="id")
threshold_transcriptome_fn=function(i){
 
    transcriptome_enrichment_df=data.frame(matrix(rep(NA, 8), ncol=8))[0,]
    colnames(transcriptome_enrichment_df)=c("CC_name", "transcriptome", "top_transcriptome", "nontop_transcriptome", "top_nontranscriptome", "nontop_nontranscriptome", "odds", "p")


    CC_list=c("CC1_rotation", "CC2_rotation", "CC3_rotation", "CC4_rotation", "CC5_rotation", "CC6_rotation", "CC7_rotation", "CC8_rotation", "CC9_rotation", "CC10_rotation")

    for (CC in CC_list){
    
        transcriptome_df3_order=transcriptome_df2[order(transcriptome_df2[, CC], decreasing=TRUE), ]
        top_list=transcriptome_df3_order[1:i, "cluster"]
        for (transcriptome_n in cluster_list){
            num_top=sum(top_list==transcriptome_n)
            all_transcriptome_list=transcriptome_df3_order$cluster
            num_all=sum(all_transcriptome_list==transcriptome_n)
            top_transcriptome_n=num_top
            nontop_transcriptome_n=num_all-num_top
            top_nontranscriptome_n=i-num_top
            nontop_nontranscriptome_n=nrow(transcriptome_df2)-i+num_top-num_all
            mat=matrix(c(top_transcriptome_n,top_nontranscriptome_n, nontop_transcriptome_n,  nontop_nontranscriptome_n), 2,2)
            testResult <- fisher.test(mat, alternative = "greater")
        
            or <- unname(testResult$estimate) 
            pValue <- unname(testResult$p.value) 
            new_df=data.frame(
            CC_name=CC,
            transcriptome=transcriptome_n,
            top_transcriptome=top_transcriptome_n,
            nontop_transcriptome=nontop_transcriptome_n,
            top_nontranscriptome=top_nontranscriptome_n,
            nontop_nontranscriptome=nontop_nontranscriptome_n,
            odds=or,
            p=pValue,
            stringsAsFactors=FALSE
            )
            transcriptome_enrichment_df=rbind(transcriptome_enrichment_df, new_df)
        }
    }



    out_f_transcriptome=paste0(out_f_Z_transcriptome,"/transcriptome_top",i,"_enrichment_ver3")
    dir.create(out_f_transcriptome, recursive=T)
    annotation_num=<number_of_annotation_groups>
    write.table(transcriptome_enrichment_df, paste0(out_f_transcriptome, "/transcriptome_enrichment.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F) 
    transcriptome_enrichment_top20=matrix(rep(NA, 20*10), ncol=10)
    colnames(transcriptome_enrichment_top20)=c("CC1_rotation", "CC2_rotation", "CC3_rotation", "CC4_rotation", "CC5_rotation", "CC6_rotation", "CC7_rotation", "CC8_rotation", "CC9_rotation", "CC10_rotation")
    rownames(transcriptome_enrichment_top20)=paste0("top_", seq(20))
    
    for (CC in CC_list){
        transcriptome_enrichment_CC=transcriptome_enrichment_df%>% filter(CC_name==CC)
        transcriptome_enrichment_CC$minuslog10p=-log10(as.numeric(transcriptome_enrichment_CC$p))
        transcriptome_enrichment_CC2=transcriptome_enrichment_CC %>% filter(minuslog10p> -log10(0.05/(annotation_num*20)))  
        transcriptome_enrichment_CC_order=transcriptome_enrichment_CC2[order(transcriptome_enrichment_CC2$odds, decreasing=TRUE),]
        transcriptome_enrichment_top20[,CC]=transcriptome_enrichment_CC_order[1:20,"transcriptome"]
    }
    transcriptome_enrichment_top20_df=as.data.frame(transcriptome_enrichment_top20)
    write.table(data.frame(top=rownames(transcriptome_enrichment_top20_df),transcriptome_enrichment_top20_df), paste0(out_f_transcriptome, "/transcriptome_enrichment_top20_odds.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F) 

      transcriptome_enrichment_df_bottom=data.frame(matrix(rep(NA, 8), ncol=8))[0,]
    colnames(transcriptome_enrichment_df_bottom)=c("CC_name", "transcriptome", "bottom_transcriptome", "nonbottom_transcriptome", "bottom_nontranscriptome", "nonbottom_nontranscriptome", "odds", "p")


    CC_list=c("CC1_rotation", "CC2_rotation", "CC3_rotation", "CC4_rotation", "CC5_rotation", "CC6_rotation", "CC7_rotation", "CC8_rotation", "CC9_rotation", "CC10_rotation")

    for (CC in CC_list){
    
        transcriptome_df3_order=transcriptome_df2[order(transcriptome_df2[, CC], decreasing=FALSE), ]
        bottom_list=transcriptome_df3_order[1:i, "cluster"]
        for (transcriptome_n in cluster_list){
            num_bottom=sum(bottom_list==transcriptome_n)
            all_transcriptome_list=transcriptome_df3_order$cluster
            num_all=sum(all_transcriptome_list==transcriptome_n)
            bottom_transcriptome_n=num_bottom
            nonbottom_transcriptome_n=num_all-num_bottom
            bottom_nontranscriptome_n=i-num_bottom
            nonbottom_nontranscriptome_n=nrow(transcriptome_df2)-i+num_bottom-num_all
            mat=matrix(c(bottom_transcriptome_n, bottom_nontranscriptome_n, nonbottom_transcriptome_n, nonbottom_nontranscriptome_n), 2,2)
            testResult <- fisher.test(mat, alternative = "greater")
        
            or <- unname(testResult$estimate) 
            pValue <- unname(testResult$p.value) 
            new_df=data.frame(
            CC_name=CC,
            transcriptome=transcriptome_n,
            bottom_transcriptome=bottom_transcriptome_n,
            nonbottom_transcriptome=nonbottom_transcriptome_n,
            bottom_nontranscriptome=bottom_nontranscriptome_n,
            nonbottom_nontranscriptome=nonbottom_nontranscriptome_n,
            odds=or,
            p=pValue,
            stringsAsFactors=FALSE
            )
            transcriptome_enrichment_df_bottom=rbind(transcriptome_enrichment_df_bottom, new_df)
        }
    }



    out_f_transcriptome_bottom=paste0(out_f_Z_transcriptome,"/transcriptome_bottom",i,"_enrichment_ver3")
    dir.create(out_f_transcriptome_bottom, recursive=T)
    write.table(transcriptome_enrichment_df_bottom, paste0(out_f_transcriptome_bottom, "/transcriptome_enrichment_bottom.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F) 
    transcriptome_enrichment_bottom20=matrix(rep(NA, 20*10), ncol=10)
    colnames(transcriptome_enrichment_bottom20)=c("CC1_rotation", "CC2_rotation", "CC3_rotation", "CC4_rotation", "CC5_rotation", "CC6_rotation", "CC7_rotation", "CC8_rotation", "CC9_rotation", "CC10_rotation")
    rownames(transcriptome_enrichment_bottom20)=paste0("bottom_", seq(20))
    
    for (CC in CC_list){
        transcriptome_enrichment_CC=transcriptome_enrichment_df_bottom%>% filter(CC_name==CC)
        transcriptome_enrichment_CC$minuslog10p=-log10(as.numeric(transcriptome_enrichment_CC$p))
        transcriptome_enrichment_CC2=transcriptome_enrichment_CC %>% filter(minuslog10p> -log10(0.05/(annotation_num*20)))  #tissue30xCC20
        transcriptome_enrichment_CC_order=transcriptome_enrichment_CC2[order(transcriptome_enrichment_CC2$odds, decreasing=TRUE),]
        transcriptome_enrichment_bottom20[,CC]=transcriptome_enrichment_CC_order[1:20,"transcriptome"]
    }
    transcriptome_enrichment_bottom20_df=as.data.frame(transcriptome_enrichment_bottom20)
    write.table(data.frame(bottom=rownames(transcriptome_enrichment_bottom20_df),transcriptome_enrichment_bottom20_df), paste0(out_f_transcriptome_bottom, "/transcriptome_enrichment_bottom20_odds.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F) 



threshold_transcriptome_fn(60)
threshold_transcriptome_fn(80)
threshold_transcriptome_fn(100)